In [2]:
# Create spark session

from pyspark.sql import SparkSession

spark = (
    # To use the PostgreSQL JDBC driver inside the container, update the Spark config to download the jar automatically via Maven coordinates.
    # Replace 'spark.jars' with 'spark.jars.packages' for the Postgres driver:
    SparkSession \
        .builder \
        .appName("Write to Multiple Sink") \
        .config("spark.streaming.stopGracefullyOnShutdown", True) \
        .config('spark.jars.packages',
                'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,org.postgresql:postgresql:42.2.20') \
        .config("spark.sql.shuffle.partitions", 8) \
        .master("local[2]") \
        .getOrCreate()
)

spark

In [3]:
# Create the kafka_df to read from kafka

kafka_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "ed-kafka:29092")
    .option("subscribe", "device-data")
    .option("startingOffsets", "earliest")
    .load()
)
kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [21]:
# Parse value from binary to string into kafka_json_df
from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))

In [22]:
# Schema for payload

from struct import Struct
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, LongType

json_schema = (
    StructType(
        [StructField("customerId", StringType(), True),
        StructField("data",StructType(
            [StructField("devices",
                        # Array of device objects
                        ArrayType(StructType([
                            StructField("deviceId", StringType(), True),
                            StructField("measure", StringType(), True),
                            StructField("status", StringType(), True),
                            StructField("temperature", LongType(), True),
                        ]), True), True)
                ]), True),
        StructField("eventID", StringType(), True),
        StructField("eventOffset", LongType(), True),
        StructField("eventPublisher", StringType(), True),
        StructField("eventTime", StringType(), True)
        ])
)

In [23]:
# Apply the schema to payload to read the data
from pyspark.sql.functions import from_json, col

streaming_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema)).selectExpr("values_json.*")

In [24]:
# lets explode the data as devices contains list/array of device reading

from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices")).drop("data")

streaming_df.printSchema()
# streaming_df.show()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventID: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [25]:
# Flatten the exploded df
from pyspark.sql.functions import col
flattened_df = (
    exploded_df
    .withColumn("deviceID", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)
flattened_df.printSchema()
# flattened_df.show()

root
 |-- customerId: string (nullable = true)
 |-- eventID: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- deviceID: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)



In [26]:
# Python function to write to multiple sinks

def device_data_output(df, batch_id):
    print("Batch ID: " + str(batch_id))

    # write to parquet
    df.write.format("parquet").mode("append").save("data/output/device_data_parquet/")
    
    # write to JDBC Postgres
    (
        df.write
        .mode("append")
        .format("jdbc")
        .option("driver", "org.postgres.Driver")
        .option("url", "jdbc:postgresql://postgres-db-1:5432/sqlpad")
        .option("dbtable", "device_data")
        .option("user", "sqlpad")
        .option("password", "sqlpad")
        .save()
    )
    # Display
    df.show()
          

In [27]:

# # Running foreachBatch
# # Write the output to console sink to check the output

(flattened_df
 .writeStream
 .foreachBatch(device_data_output)
 .outputMode("append")
 .trigger(processingTime='10 seconds')
 .option("checkpointLocation", "checkpoint_dir_kafka")
 .start()
 .awaitTermination())

StreamingQueryException: Query [id = b140e4b8-c2de-47e6-939e-25bfd16bfdc2, runId = 8228c6f4-4cb1-48e0-9a77-bbb352cc60e1] terminated with exception: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/spark/python/pyspark/sql/utils.py", line 272, in call
    raise e
  File "/spark/python/pyspark/sql/utils.py", line 269, in call
    self.func(DataFrame(jdf, self.session), batch_id)
  File "/tmp/ipykernel_132/3202458249.py", line 19, in device_data_output
    .save()
  File "/spark/python/pyspark/sql/readwriter.py", line 966, in save
    self._jwrite.save()
  File "/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/java_gateway.py", line 1321, in __call__
    return_value = get_return_value(
  File "/spark/python/pyspark/sql/utils.py", line 190, in deco
    return f(*a, **kw)
  File "/s